In [6]:
from bs4 import BeautifulSoup as bs
import requests as req
from pprint import pprint
import lxml
import json

In [ ]:
all_articles_url = []

search_content = input("請輸入要查詢的關鍵字(食材或料理名稱):")
search_keyword = 'tw/搜尋/'
base_url = "https://cookpad.com/"
url = base_url + search_keyword + search_content

res = req.get(url)

print(res.status_code)
soup = bs(res.text, 'lxml')

# CSS 選擇器中的冒號 (:) 被 BeautifulSoup 或 Parsel (Scrapy) 誤認為是 CSS 虛擬類別（Pseudo-class，例如 :hover 或 :nth-child）
# 在 CSS 選擇器中，必須使用 反斜線 (\) 來轉義（Escape）特殊字元。但在 Python 字串中，反斜線本身也需要被轉義，因此最快的方法是使用雙反斜線 \\
css_selector = 'li.ranked-list__item.xs\\:mb-sm'
articles = soup.select(css_selector)

for aurl in articles:
    title_css_selector = 'a.block-link__main'
    title_text = aurl.select_one(title_css_selector).text

    if title_text.strip():
        recipe_url = aurl.a['href']
        all_articles_url.append(base_url+recipe_url)

for aurl in all_articles_url:
    print(aurl)

200
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/24552577
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/23950614
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/22660023
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/22613976
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/22601147
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/22557636
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/17391340
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/24292907
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/17276119
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/24032306
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/23954013
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/23937404
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/22578367
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/16979493
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/17199802
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/16858531
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/16562239
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/16439717
https://cookpad.com//tw/%E9%A3%9F%E8%AD%9C/16343444
https://

In [23]:
print(len(all_articles_url))

30


In [1]:
# 爬取動態載入頁面的套件
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException

import pandas as pd
from time import sleep

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(10) # 隱性等待
driver.get('https://cookpad.com/tw/搜尋/素食%20茄子')

# 初始化 ActionChains
action = ActionChains(driver)

In [6]:
driver.quit()
print("驅動程式已關閉")

驅動程式已關閉


In [2]:
# 初始化前一次抓取的食譜數量
last_count = 0

while True:
    # 取得目前頁面上的所有食譜元素
    recipe_list_whole = driver.find_elements(By.CSS_SELECTOR, 'li.ranked-list__item')
    current_count = len(recipe_list_whole)
    print(current_count)

    # 判斷新資料是否有載入
    if current_count == last_count:
        print("數量未增加，故終止迴圈(已到達底部)")
        break

    # 找出目前清單的「最後一個元素」
    last_element = recipe_list_whole[-1]

    # 使用ActionChains 滾動到最後一個元素
    action.scroll_to_element(last_element).perform()
    
    # 等待載入
    sleep(2)
    
    # 更新狀態 : 將目前的數量記錄下來，供下一輪比對
    last_count = current_count

30
60
90
99
99
數量未增加，故終止迴圈(已到達底部)


In [7]:
print(len(recipe_list_whole))

951


In [5]:
# 建立一個空的容器，放置食譜資訊
recipe_list = []

# 遍歷所有食譜元素並提取詳細的食譜資訊
for i, element in enumerate(recipe_list_whole):
    try:
        # 取得食譜標題
        title = element.find_element(By.CSS_SELECTOR, 'a.block-link__main').text

        # 取得製作時間和份量
        cooking_time = "未提供"
        servings = "未提供"

        info_items = element.find_elements(By.CSS_SELECTOR, 'li.inline.mr-rg span.mise-icon-text')
        
        for item in info_items:
            item_text = item.text.strip()

            if '分鐘' in item_text or '小時' in item_text:
                cooking_time = item_text
            elif '人' in item_text:
                servings = item_text

        # 取得食譜連結
        url = element.find_element(By.CSS_SELECTOR, 'li.ranked-list__item a.block-link__main')
        recipe_link = url.get_attribute('href')

        # 取得食譜圖片
        img_element = element.find_element(By.CSS_SELECTOR, 'div.flex-none picture img.h-full')

        img_url = img_element.get_attribute('src')

        # 處理 Lazy Loading 情況：如果 src 是預載圖或空的，嘗試抓取 data-src
        if not img_url or 'data:image' in img_url:
            img_url = img_element.get_attribute('data-src')

        # print(f"[{i+1}] 食譜標題: {title}")
        # print(f"     圖片連結: {img_url}")
        # print("-" * 50)

        # 整合成一筆資料
        recipe_dict = {
            "title": title,
            "cooking_time": cooking_time,
            "servings": servings,
            "recipe_link": recipe_link,
            "img_url": img_url
        }

        recipe_list.append(recipe_dict)
    except Exception as e:
        print(e)

    print(f"{i+1}, {title}, 製作時間:{cooking_time}, 份量:{servings}, 食譜連結:{recipe_link}, 圖片連結:{img_url}")

1, (烹大師香菇風味) 素炒三杯時蔬-杏鮑菇,茄子,青椒, 製作時間:25分鐘, 份量:2人份, 食譜連結:https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/24552577, 圖片連結:https://img-global.cpcdn.com/recipes/fcee5b0ddebf02c7/120x160cq50/%E7%83%B9%E5%A4%A7%E5%B8%AB%E9%A6%99%E8%8F%87%E9%A2%A8%E5%91%B3-%E7%B4%A0%E7%82%92%E4%B8%89%E6%9D%AF%E6%99%82%E8%94%AC-%E6%9D%8F%E9%AE%91%E8%8F%87%E8%8C%84%E5%AD%90%E9%9D%92%E6%A4%92-%E9%A3%9F%E8%AD%9C%E6%88%90%E5%93%81%E7%85%A7%E7%89%87.jpg
2, 香料茄子馬鈴薯（純素）, 製作時間:40分鐘, 份量:5人份, 食譜連結:https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/23950614, 圖片連結:https://img-global.cpcdn.com/recipes/97300b86783f9267/120x160f0.469949_0.5_1.0q50/%E9%A6%99%E6%96%99%E8%8C%84%E5%AD%90%E9%A6%AC%E9%88%B4%E8%96%AF%E7%B4%94%E7%B4%A0-%E9%A3%9F%E8%AD%9C%E6%88%90%E5%93%81%E7%85%A7%E7%89%87.jpg
3, 九層塔茄子炒豆包和豆腐（純素）, 製作時間:30分鐘, 份量:5人份, 食譜連結:https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/22660023, 圖片連結:https://img-global.cpcdn.com/recipes/2482ba1b52d64ef5/120x160f0.512295_0.5_1.0q50/%E4%B9%9D%E5%B1%A4%E5%A1%94%E8%8C%84%E5%AD%90%E7%82%92%E8%B1%86%E5%8C%85%E5%9

## 程式碼片段測試區域

In [28]:
test_ele1 = recipe_list_whole[9]
test_ele2 = recipe_list_whole[10]
test_ele3 = recipe_list_whole[11]

try:
    cooking_time_elem = test_ele3.find_element(By.XPATH, './/span[contains(@class, "mise-icon-text") and (contains(text(), "分鐘") or contains(text(), "小時") or .//svg[contains(@data-icon, "clock")])]')
    cooking_time = cooking_time_elem.text
except NoSuchElementException:
    cooking_time = "未提供"

try:
    servings_elem = test_ele3.find_element(By.XPATH, './/span[contains(@class, "mise-icon-text") and (contains(text(), "人") or .//svg[contains(@data-icon, "user")])]')
    servings = servings_elem.text
except NoSuchElementException:
    servings = "未提供"

print(f"此道料理的製作時間: {cooking_time}, 份量: {servings}")

此道料理的製作時間: 未提供, 份量: 未提供


In [33]:
test_ele1 = recipe_list_whole[9]
test_ele2 = recipe_list_whole[10]
test_ele3 = recipe_list_whole[11]

cooking_time = "未提供"
servings = "未提供"

tags = test_ele1.find_elements(By.CSS_SELECTOR, 'li.inline.mr-rg span.mise-icon-text')

for tag in tags:
    text = tag.text.strip()
    print(text)
    if '分鐘' in text or '小時' in text:
        cooking_time = text
    elif '人' in text:
        servings = text

print(f"時間: {cooking_time}, 份量: {servings}")

20分鐘
6人
時間: 20分鐘, 份量: 6人


In [36]:
test_ele1 = recipe_list_whole[10]

url = test_ele1.find_element(By.CSS_SELECTOR, 'li.ranked-list__item a.block-link__main')
print(url.get_attribute('href'))

https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/15086348
